# Baseline (NPZ): Official ResNet1d Wang

Important: the official README reports AUC 0.936 on diagnostic statements (and 0.919 on all statements).
Here we train the official Wang et al. ResNet1d on superdiagnostic (5 classes) using your NPZ.

In [30]:
# STEP 1 — Install
!pip install -q scikit-learn
print("✅ Ready")

✅ Ready


In [ ]:
# STEP 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = '/content/drive/MyDrive/ecg-multigraph-lab'
CKPT_DIR = f'{PROJECT_ROOT}/checkpoints'
PRED_DIR = f'{PROJECT_ROOT}/predictions'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PRED_DIR,  exist_ok=True)
print("✅ Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted


In [ ]:
# STEP 3 — Load data
import os
from pathlib import Path

# Try common locations first
candidates = [
    '/content/drive/MyDrive/ecg-multigraph-lab/data_preprocessed/ptbxl_sota_100hz_diagnostic_superclass.npz',
    '/content/drive/MyDrive/ecg-multigraph-lab/ptbxl_sota_100hz_diagnostic_superclass.npz',
]
NPZ_PATH = None
for p in candidates:
    if os.path.exists(p):
        NPZ_PATH = p
        break

if NPZ_PATH is None:
    print('File not found in common locations, searching Drive...')
    drive_root = Path('/content/drive/MyDrive')
    matches = list(drive_root.rglob('ptbxl_sota_100hz_diagnostic_superclass.npz'))
    if not matches:
        raise FileNotFoundError('Could not find NPZ under /content/drive/MyDrive')
    NPZ_PATH = str(matches[0])
    print('Found candidates:')
    for m in matches:
        print(' -', m)
else:
    print('Found:', NPZ_PATH)

data = np.load(NPZ_PATH, allow_pickle=True)
print("NPZ keys:", list(data.files))

# Format 1: X_train / y_train
if all(k in data.files for k in ['X_train','y_train','X_val','y_val','X_test','y_test']):
    X_train, y_train = data['X_train'], data['y_train']
    X_val,   y_val   = data['X_val'],   data['y_val']
    X_test,  y_test  = data['X_test'],  data['y_test']
    print("✅ Format: X_train/y_train")

# Format 2: signals / labels / splits
elif all(k in data.files for k in ['signals','labels','splits']):
    splits = data['splits'].astype(str)
    X, y   = data['signals'], data['labels']
    X_train, y_train = X[splits=='train'], y[splits=='train']
    X_val,   y_val   = X[splits=='val'],   y[splits=='val']
    X_test,  y_test  = X[splits=='test'],  y[splits=='test']
    print("✅ Format: signals/labels/splits")

else:
    raise KeyError(f"Неизвестный формат NPZ. Ключи: {list(data.files)}")

# class names
if 'class_names' in data.files:
    class_names = [c.decode() if isinstance(c, bytes) else str(c)
                   for c in data['class_names']]
else:
    class_names = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

# dtype
X_train = X_train.astype(np.float32); y_train = y_train.astype(np.float32)
X_val   = X_val.astype(np.float32);   y_val   = y_val.astype(np.float32)
X_test  = X_test.astype(np.float32);  y_test  = y_test.astype(np.float32)

# ── Нормализация ─────────────────────────────────────────
print(f"\nДо нормализации:")
print(f"  mean={X_train.mean():.4f}, std={X_train.std():.4f}, "
      f"min={X_train.min():.4f}, max={X_train.max():.4f}")

mean = X_train.mean()
std  = X_train.std()

if std < 1e-8:
    raise ValueError(f"std слишком мал: {std}. Проверь данные.")

X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

print(f"После нормализации:")
print(f"  mean={X_train.mean():.4f}, std={X_train.std():.4f}, "
      f"min={X_train.min():.4f}, max={X_train.max():.4f}")

print(f"\nTrain : {X_train.shape}  labels: {y_train.shape}")
print(f"Val   : {X_val.shape}   labels: {y_val.shape}")
print(f"Test  : {X_test.shape}  labels: {y_test.shape}")
print(f"Classes: {class_names}")

# ── pos_weight для BCEWithLogitsLoss ─────────────────────
import torch

pos_counts = y_train.sum(axis=0)                 # [5] сколько 1 по каждому классу
neg_counts = len(y_train) - pos_counts           # [5] сколько 0

alpha = 0.4
pos_weight_np = ((neg_counts / (pos_counts + 1e-6)) ** alpha).astype(np.float32)
print("pos_weight:", pos_weight_np)

pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32)

Found: /content/drive/MyDrive/ecg-multigraph-lab/data_preprocessed/ptbxl_sota_100hz_diagnostic_superclass.npz
NPZ keys: ['signals', 'labels', 'splits', 'ecg_ids', 'class_names', 'sampling_rate']
✅ Format: signals/labels/splits

До нормализации:
  mean=-0.0000, std=0.9998, min=-10.0000, max=10.0000
После нормализации:
  mean=-0.0000, std=1.0000, min=-10.0016, max=10.0016

Train : (17418, 12, 1000)  labels: (17418, 5)
Val   : (2183, 12, 1000)   labels: (2183, 5)
Test  : (2198, 12, 1000)  labels: (2198, 5)
Classes: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
pos_weight: [1.1082711 1.5472045 1.6046607 1.642622  2.2050216]


In [ ]:
# STEP 4 — DataLoaders (без WeightedRandomSampler)
from torch.utils.data import Dataset, DataLoader

class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


def make_loader(X, y, batch_size=128, shuffle=True):
    ds = ECGDataset(X, y)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,      # чтобы не падало в Colab
        pin_memory=False,
    )

# batch_size=128
train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   batch_size=128, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  batch_size=128, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train batches: 137 | Val: 18 | Test: 18


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── из basic_conv1d.py: AdaptiveConcatPool + Head ────────────────────────
class AdaptiveConcatPool1d(nn.Module):
    """Concat avg+max pool — оригинал из basic_conv1d.py"""
    def __init__(self):
        super().__init__()
        self.ap = nn.AdaptiveAvgPool1d(1)
        self.mp = nn.AdaptiveMaxPool1d(1)
    def forward(self, x):
        return torch.cat([self.mp(x), self.ap(x)], dim=1)  # [B, 2*C]

def create_head1d(nf, nc, ps=0.5):
    """Head: ConcatPool → Flatten → BN → Drop → Linear → BN → Drop → Linear"""
    return nn.Sequential(
        AdaptiveConcatPool1d(),
        nn.Flatten(),
        nn.BatchNorm1d(2 * nf),
        nn.Dropout(ps / 2),
        nn.Linear(2 * nf, 512, bias=False),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(ps),
        nn.Linear(512, nc),
    )

# ── из resnet1d-2.py: conv helper ────────────────────────────────────────
def conv1d_pad(in_planes, out_planes, stride=1, kernel_size=3):
    """Conv1d с same-padding (kernel_size - 1) // 2 — как в оригинале"""
    return nn.Conv1d(
        in_planes, out_planes,
        kernel_size=kernel_size,
        stride=stride,
        padding=(kernel_size - 1) // 2,
        bias=False,
    )

# ── из resnet1d-2.py: BasicBlock1d ───────────────────────────────────────
class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1,
                 kernel_size=[5, 3], downsample=None):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size // 2 + 1]

        self.conv1     = conv1d_pad(inplanes, planes, stride=stride, kernel_size=kernel_size[0])
        self.bn1       = nn.BatchNorm1d(planes)
        self.relu      = nn.ReLU(inplace=True)
        self.conv2     = conv1d_pad(planes, planes, kernel_size=kernel_size[1])
        self.bn2       = nn.BatchNorm1d(planes)
        self.downsample = downsample
        self.stride    = stride

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return self.relu(out)

# ── из resnet1d-2.py: ResNet1d backbone ──────────────────────────────────
class ResNet1dWang(nn.Module):
    """
    resnet1d_wang — ТОЧНАЯ архитектура из PTB-XL бенчмарка.
    Файл: helme/ecg_ptbxl_benchmarking/code/models/resnet1d.py

    Параметры (из функции resnet1d_wang()):
      BasicBlock1d, layers=[1,1,1], inplanes=128,
      kernel_size=[5,3], kernel_size_stem=7,
      stride_stem=1, pooling_stem=False

    Голова: AdaptiveConcatPool1d → Flatten → BN → Drop → Linear × 2
    """
    def __init__(self, num_classes=5, input_channels=12, ps_head=0.5):
        super().__init__()
        inplanes       = 128
        kernel_size    = [5, 3]
        kernel_size_stem = 7
        stride_stem    = 1

        # Stem
        self.stem = nn.Sequential(
            conv1d_pad(input_channels, inplanes,
                       stride=stride_stem, kernel_size=kernel_size_stem),
            nn.BatchNorm1d(inplanes),
            nn.ReLU(inplace=True),
            # pooling_stem=False → нет MaxPool
        )

        # 3 стадии, [1, 1, 1] блок каждая, stride=2 начиная со 2-й
        self.layer1 = self._make_layer(inplanes, inplanes, blocks=1,
                                       stride=1, kernel_size=kernel_size)
        self.layer2 = self._make_layer(inplanes, inplanes, blocks=1,
                                       stride=2, kernel_size=kernel_size)
        self.layer3 = self._make_layer(inplanes, inplanes, blocks=1,
                                       stride=2, kernel_size=kernel_size)
        self.inplanes = inplanes

        # Head
        self.head = create_head1d(inplanes, nc=num_classes, ps=ps_head)

    def _make_layer(self, in_planes, planes, blocks, stride, kernel_size):
        downsample = None
        if stride != 1 or in_planes != planes * BasicBlock1d.expansion:
            downsample = nn.Sequential(
                nn.Conv1d(in_planes, planes * BasicBlock1d.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes * BasicBlock1d.expansion),
            )
        layers = [BasicBlock1d(in_planes, planes, stride, kernel_size, downsample)]
        in_planes = planes * BasicBlock1d.expansion
        for _ in range(1, blocks):
            layers.append(BasicBlock1d(in_planes, planes,
                                       kernel_size=kernel_size))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.head(x)   # голова включает pooling + linear


# ── Проверка ─────────────────────────────────────────────────────────────
_m   = ResNet1dWang(num_classes=5, input_channels=12)
_x   = torch.randn(4, 12, 1000)
_out = _m(_x)
_p   = sum(p.numel() for p in _m.parameters())
print(f"Output : {_out.shape}")   # torch.Size([4, 5])
print(f"Params : {_p:,}")
print("✅ OK — точная архитектура resnet1d_wang из PTB-XL бенчмарка")

Output : torch.Size([4, 5])
Params : 574,213
✅ OK — точная архитектура resnet1d_wang из PTB-XL бенчмарка


In [ ]:
# STEP 6 — Metrics
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def fmax_score(y_true, y_prob, n_thresh=201):
    """Macro-F1 максимизированный по порогу — стандарт PTB-XL бенчмарка."""
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0, 1, n_thresh):
        f1 = f1_score(y_true, (y_prob >= t).astype(int),
                      average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_f1, best_t

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for X, y in loader:
        probs = torch.sigmoid(model(X.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return {
        'macro_auc':   roc_auc_score(labels, probs, average='macro'),
        'macro_auprc': average_precision_score(labels, probs, average='macro'),
        'fmax':        fmax_score(labels, probs),
        'probs':       probs,
        'labels':      labels,
    }

print("✅ Metrics OK")

✅ Metrics OK


In [ ]:
# STEP 7 — Training loop (Adam + OneCycleLR + pos_weight)
from torch.optim.lr_scheduler import OneCycleLR

def train_one_epoch(model, loader, optimizer, criterion, device, scheduler):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        scheduler.step()           # OneCycleLR шагает каждый batch
        total_loss += loss.item()
    return total_loss / len(loader)


def train_model(model, train_loader, val_loader,
                epochs=80, lr=1e-3,
                patience=20, save_path='best.pth', device='cuda'):

    model = model.to(device)

    # pos_weight переносим на нужный девайс
    pw = pos_weight.to(device)

    # BCEWithLogitsLoss с pos_weight — учёт дисбаланса классов
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    # Adam — как в большинстве PTB-XL реализаций
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # OneCycleLR — приближение fastai fit_one_cycle
    scheduler = OneCycleLR(
        optimizer,
        max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.3,           # 30% эпох — разгон, 70% — спуск
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e4,
        cycle_momentum=True,
        base_momentum=0.85,
        max_momentum=0.95,
    )

    best_auc   = 0.0
    no_improve = 0

    for epoch in range(1, epochs + 1):
        loss    = train_one_epoch(model, train_loader, optimizer,
                                  criterion, device, scheduler)
        metrics = evaluate(model, val_loader, device)
        val_auc  = metrics['macro_auc']
        val_aupr = metrics['macro_auprc']

        mark = ''
        if val_auc > best_auc:
            best_auc, no_improve = val_auc, 0
            torch.save(model.state_dict(), save_path)
            mark = ' ✅'
        else:
            no_improve += 1

        cur_lr = optimizer.param_groups[0]['lr']
        print(f"Ep {epoch:3d} | loss={loss:.4f} | "
              f"AUC={val_auc:.4f} | AUPRC={val_aupr:.4f} | "
              f"lr={cur_lr:.1e}{mark}")

        if no_improve >= patience:
            print(f"⏹ Early stop @ epoch {epoch}")
            break

    print(f"\n{'='*50}\nBest Val AUC: {best_auc:.4f}\n{'='*50}")
    return best_auc

print("✅ Training loop OK (pos_weight)")

✅ Training loop OK (pos_weight)


In [ ]:
# STEP 8 — Run training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {device}")
if device == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
if device == 'cuda':
    torch.cuda.manual_seed_all(42)

model = ResNet1dWang(num_classes=len(class_names),
                     input_channels=X_train.shape[1])
print(f"Params : {sum(p.numel() for p in model.parameters()):,}")

CKPT_PATH = f'{CKPT_DIR}/resnet1d_wang_baseline.pth'

best_auc = train_model(
    model,
    train_loader,
    val_loader,
    epochs    = 80,      # официальный бенчмарк: ~50 эпох с OneCycleLR
    lr        = 1e-3,
    patience  = 15,
    save_path = CKPT_PATH,
    device    = device,
)

Device : cuda
GPU    : Tesla T4
Params : 574,213
Ep   1 | loss=0.6426 | AUC=0.8580 | AUPRC=0.6653 | lr=4.4e-05 ✅
Ep   2 | loss=0.4831 | AUC=0.8765 | AUPRC=0.7115 | lr=5.6e-05 ✅
Ep   3 | loss=0.4329 | AUC=0.8871 | AUPRC=0.7328 | lr=7.7e-05 ✅
Ep   4 | loss=0.4045 | AUC=0.8944 | AUPRC=0.7411 | lr=1.0e-04 ✅
Ep   5 | loss=0.3872 | AUC=0.8971 | AUPRC=0.7441 | lr=1.4e-04 ✅
Ep   6 | loss=0.3782 | AUC=0.8981 | AUPRC=0.7437 | lr=1.8e-04 ✅
Ep   7 | loss=0.3685 | AUC=0.8959 | AUPRC=0.7455 | lr=2.3e-04
Ep   8 | loss=0.3645 | AUC=0.9022 | AUPRC=0.7538 | lr=2.8e-04 ✅
Ep   9 | loss=0.3596 | AUC=0.9005 | AUPRC=0.7484 | lr=3.4e-04
Ep  10 | loss=0.3543 | AUC=0.9045 | AUPRC=0.7583 | lr=4.0e-04 ✅
Ep  11 | loss=0.3479 | AUC=0.8999 | AUPRC=0.7488 | lr=4.6e-04
Ep  12 | loss=0.3409 | AUC=0.9058 | AUPRC=0.7659 | lr=5.2e-04 ✅
Ep  13 | loss=0.3382 | AUC=0.9057 | AUPRC=0.7609 | lr=5.8e-04
Ep  14 | loss=0.3376 | AUC=0.9030 | AUPRC=0.7608 | lr=6.4e-04
Ep  15 | loss=0.3323 | AUC=0.9094 | AUPRC=0.7676 | lr=7.0e-04 ✅
E

In [ ]:
# STEP 9 — Final test evaluation
# Загружаем ЛУЧШИЙ чекпоинт (не последний!)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model = model.to(device)

test_m = evaluate(model, test_loader, device)
fmax, fmax_t = test_m['fmax']

print("=" * 50)
print("  BASELINE resnet1d_wang — TEST RESULTS")
print("=" * 50)
print(f"  Macro AUROC  : {test_m['macro_auc']:.4f}")
print(f"  Macro AUPRC  : {test_m['macro_auprc']:.4f}")
print(f"  Fmax         : {fmax:.4f}  (@ t={fmax_t:.2f})")
print("-" * 50)
print("  Per-class AUROC:")
for i, name in enumerate(class_names):
    auc_i = roc_auc_score(test_m['labels'][:, i], test_m['probs'][:, i])
    print(f"    {name:<6}: {auc_i:.4f}")
print("=" * 50)

  BASELINE resnet1d_wang — TEST RESULTS
  Macro AUROC  : 0.9106
  Macro AUPRC  : 0.7669
  Fmax         : 0.7055  (@ t=0.43)
--------------------------------------------------
  Per-class AUROC:
    NORM  : 0.9414
    MI    : 0.9280
    STTC  : 0.9273
    CD    : 0.9143
    HYP   : 0.8420


In [ ]:
# STEP 10 — Save to Drive
np.save(f'{PRED_DIR}/baseline_test_probs.npy',  test_m['probs'])
np.save(f'{PRED_DIR}/baseline_test_labels.npy', test_m['labels'])

with open(f'{PRED_DIR}/baseline_metrics.txt', 'w') as f:
    f.write(f"Model       : resnet1d_wang (PTB-XL official benchmark)\n"
            f"Macro AUROC : {test_m['macro_auc']:.4f}\n"
            f"Macro AUPRC : {test_m['macro_auprc']:.4f}\n"
            f"Fmax        : {fmax:.4f} @ t={fmax_t:.2f}\n")

print("✅ Saved:")
print(f"   {CKPT_PATH}")
print(f"   {PRED_DIR}/baseline_test_probs.npy")
print(f"   {PRED_DIR}/baseline_metrics.txt")
print("\n🎯 Baseline готов → следующий шаг: Lead-wise ResNet + GNN Head")

✅ Saved:
   /content/drive/MyDrive/ecg-multigraph-lab/checkpoints/resnet1d_wang_baseline.pth
   /content/drive/MyDrive/ecg-multigraph-lab/predictions/baseline_test_probs.npy
   /content/drive/MyDrive/ecg-multigraph-lab/predictions/baseline_metrics.txt

🎯 Baseline готов → следующий шаг: Lead-wise ResNet + GNN Head
